# Project to grid

Following preparation of SV gauss coefficient time series data sets with the chosen temporal/ spatial sampling parameters.

Now, must project the data at each time step from gauss coefficient to a physical grid.

## Preamble

In [2]:
# setting up file path for imports
import sys
import os
from pathlib import Path

cwd = Path(os.getcwd())
print(cwd)
PROJECT_ROOT = cwd.resolve().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

# installing relevant libraries
import chaosmagpy as cp
import numpy as np
import matplotlib.pyplot as plt
import h5py

# Project paths / utilities
from src.msc_thesis.paths import *
from src.msc_thesis.synSetup import *
from src.msc_thesis.synUtils import *
from src.msc_thesis.synDMD import *

/home/elliott/projects/msc_thesis_project/Last_stretch/Data_prep
['/home/elliott/projects/msc_thesis_project', '/home/elliott/miniforge3/envs/msc_thesis/lib/python311.zip', '/home/elliott/miniforge3/envs/msc_thesis/lib/python3.11', '/home/elliott/miniforge3/envs/msc_thesis/lib/python3.11/lib-dynload', '', '/home/elliott/miniforge3/envs/msc_thesis/lib/python3.11/site-packages', '/home/elliott/projects/msc_thesis_project']
['/home/elliott/projects/msc_thesis_project', '/home/elliott/projects/msc_thesis_project', '/home/elliott/miniforge3/envs/msc_thesis/lib/python311.zip', '/home/elliott/miniforge3/envs/msc_thesis/lib/python3.11', '/home/elliott/miniforge3/envs/msc_thesis/lib/python3.11/lib-dynload', '', '/home/elliott/miniforge3/envs/msc_thesis/lib/python3.11/site-packages', '/home/elliott/projects/msc_thesis_project']


/home/elliott/miniforge3/envs/msc_thesis/lib/python3.11/site-packages/chaosmagpy/model_utils.py:795: UserWarning: Input coordinates include the poles.
  warnings.warn('Input coordinates include the poles.')


['/home/elliott/projects/msc_thesis_project', '/home/elliott/projects/msc_thesis_project', '/home/elliott/projects/msc_thesis_project', '/home/elliott/miniforge3/envs/msc_thesis/lib/python311.zip', '/home/elliott/miniforge3/envs/msc_thesis/lib/python3.11', '/home/elliott/miniforge3/envs/msc_thesis/lib/python3.11/lib-dynload', '', '/home/elliott/miniforge3/envs/msc_thesis/lib/python3.11/site-packages', '/home/elliott/projects/msc_thesis_project']
C has shape dimensions: (148, 440, 440)


## Loading in modified gauss time series

In [3]:
# LOADING IN DATA

# CHAOS gauss coeffs
gnm_chaos_full = CHAOS_Full_SV_Record_Obtain()

# Synthetic gauss coeffs
mode_numbers = np.arange(1, 63, 10)
mode_numbers = [str(mode_number) for mode_number in mode_numbers]
gnm_syn_ideal_full, gnm_syn_full, syn_suite_info = Synthetic_Full_SV_Record_Obtain(mode_numbers, times_res_relative)

# PREPARING DATA

# applying just to ideal synthetic record - equivalent to others
gnm_syn_ideal = Gauss_Record_Prepare(gnm_syn_ideal_full)
gnm_syn = Gauss_Record_Prepare(gnm_syn_full)
gnm_chaos = Gauss_Record_Prepare(gnm_chaos_full)


Full CHAOS-8.6 timespan is: [1997.1 2026.6]


Loading combined modes: 100%|██████████| 7/7 [00:00<00:00, 14.46it/s]


## Projecting data to grid

In [8]:
# latitude/ longitude parameters
d_degree = 5 # cell width in degrees
colatitude = np.arange(0, 181, d_degree)
latitude = colatitude - 90
longitude = np.arange(0, 360, d_degree)
# theta, phi (radians):
theta = np.deg2rad(colatitude)
phi = np.deg2rad(longitude)

# latitude weighting variables
phi_grid, theta_grid = np.meshgrid(phi, theta)
W_theta = np.sin(theta)
W2D = np.sin(theta_grid)
W2D_norm = W2D / np.sum(W2D)

# getting dimensionality
state_shape = (len(theta), len(phi)) # snapshot shape used globally
n_points_globally = len(theta) * len(phi) # number of points in grid


In [11]:
# for projecting gauss coefficients onto gridded data form
# these need to be in degrees for chaosmagpy
theta_v = np.rad2deg(theta_grid.copy().ravel())
phi_v = np.rad2deg(phi_grid.copy().ravel())
radius_v = np.full(len(phi_v), r_cmb)

# gauss -> physical forward operators
# using nmax = 15
A_15_r, A_15_t, A_15_p = cp.model_utils.design_gauss(
    radius_v, theta_v, phi_v, nmax=15, source="internal"
)

A_15_dict = {"r": A_15_r, "theta": A_15_t, "phi": A_15_p}

/home/elliott/miniforge3/envs/msc_thesis/lib/python3.11/site-packages/chaosmagpy/model_utils.py:795: UserWarning: Input coordinates include the poles.
  warnings.warn('Input coordinates include the poles.')


In [ ]:
print(np.shape(A_15_r))
print(len(theta)*len(phi))

(2664, 440)
2664


In [13]:
# projecting a gauss coefficient data set
sv_syn = A_15_r @ gnm_syn.T

In [17]:
print(np.shape(np.reshape(sv_syn[:,0], shape=state_shape)))

(37, 72)
